# Notebook 05 — Risk Scoring (FR6)
**Team 7 — Mansi & Samyak**

Builds per-customer risk features, trains Logistic Regression vs Random Forest,
picks the better model, and saves `risk_model.pkl` + `final_dataset_with_risk.csv`.

In [13]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print('✅ Imports done')


✅ Imports done


## 1. Load Data

In [14]:
RAW_PATH       = r'C:\Users\mansi.apet\Downloads\AI_SUBSCRIPTION_SYSTEM_FINAL\data\transactions_raw.csv'
RECURRING_PATH = r'C:\Users\mansi.apet\Downloads\AI_SUBSCRIPTION_SYSTEM_FINAL\data\transactions_patterns.csv'

df_raw = pd.read_csv(RAW_PATH, parse_dates=['Date'])
df_rec = pd.read_csv(RECURRING_PATH)[['TransactionID','Is_Recurring']]

df = df_raw.merge(df_rec, on='TransactionID', how='left')
df['Is_Recurring']  = df['Is_Recurring'].fillna(False)
df = df.rename(columns={'SubscriptionFlag': 'is_subscription'})

print(f'Rows: {len(df):,}  |  Customers: {df["CustomerID"].nunique():,}')
df.head()


Rows: 137,806  |  Customers: 1,200


,CustomerID,CustomerName,TransactionID,Date,Description,Amount,Balance,Merchant,TransactionType,is_subscription,Status,Frequency,Is_Recurring
0,CUST100000,Karan Verma,TXN28728463,2023-01-02,MONTHLY SALARY CREDIT,143839.29,223750.99,Employer,CREDIT,0,SUCCESS,monthly,0
1,CUST100000,Karan Verma,TXN31429110,2023-01-14,NETFLIX.COM MONTHLY SUB,661.05,NaN,Netflix,DEBIT,1,SUCCESS,monthly,1
2,CUST100000,Karan Verma,TXN23756669,2023-02-02,MONTHLY SALARY CREDIT,143839.29,367590.28,Employer,CREDIT,0,SUCCESS,monthly,0
3,CUST100000,Karan Verma,TXN55176955,2023-02-14,NETFLIX.COM MONTHLY SUB,654.12,2667703.75,Netflix,DEBIT,1,SUCCESS,monthly,1
4,CUST100000,Karan Verma,TXN83197857,2023-03-02,MONTHLY SALARY CREDIT,143839.29,511429.57,Employer,CREDIT,0,SUCCESS,monthly,0


## 2. Build Per-Customer Features

In [15]:
FEATURES = [
    'total_spent','avg_amount','num_transactions',
    'num_subscriptions','num_recurring',
    'num_failed','failed_ratio',
    'avg_balance','min_balance'
]

customer_df = df.groupby('CustomerID').agg(
    total_spent       = ('Amount',        'sum'),
    avg_amount        = ('Amount',        'mean'),
    num_transactions  = ('Amount',        'count'),
    num_subscriptions = ('is_subscription','sum'),
    num_recurring     = ('Is_Recurring',  'sum'),
    num_failed        = ('Status',        lambda x: (x=='FAILED').sum()),
    avg_balance       = ('Balance',       'mean'),
    min_balance       = ('Balance',       'min'),
).reset_index()

customer_df['failed_ratio'] = customer_df['num_failed'] / customer_df['num_transactions']

print(f'Customer feature matrix: {customer_df.shape}')
customer_df.head()


Customer feature matrix: (1200, 10)


,CustomerID,total_spent,avg_amount,num_transactions,num_subscriptions,num_recurring,num_failed,avg_balance,min_balance,failed_ratio
0,CUST100000,2667392.31,49396.153889,54,18,18,0,2.235890e+06,223750.99,0.000000
1,CUST100001,1216184.65,12537.986082,97,36,36,2,1.072695e+06,239937.02,0.020619
2,CUST100002,1267672.41,9531.371504,133,96,94,5,1.279399e+06,254439.03,0.037594
3,CUST100003,2233735.69,13874.134720,161,93,92,4,2.017695e+06,262354.54,0.024845
4,CUST100004,2632983.69,15488.139353,170,130,130,10,2.336385e+06,228736.88,0.058824


## 3. Create Risk Labels

In [16]:
# Label = 1 if: more than 2 failed debits, OR
#               total spend in top 10%, OR avg amount in top 10%
T_SPENT = customer_df['total_spent'].quantile(0.9)
T_AMT   = customer_df['avg_amount'].quantile(0.9)

customer_df['risk_label'] = (
    (customer_df['num_failed'] > 2) |
    (customer_df['total_spent'] > T_SPENT) |
    (customer_df['avg_amount']  > T_AMT)
).astype(int)

print('Risk label distribution:')
print(customer_df['risk_label'].value_counts())


Risk label distribution:
risk_label
1    898
0    302
Name: count, dtype: int64


## 4. Train Both Models

In [17]:
X = customer_df[FEATURES]
y = customer_df['risk_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_s, y_train)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.07,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_s, y_train)     # if using scaled features
print('✅ All models trained')


✅ All models trained


## 5. Compare Models

In [18]:
def evaluate(name, model, X, y):
    y_pred = model.predict(X)
    return {'Model': name,
            'Accuracy':  round(accuracy_score(y, y_pred),  4),
            'Precision': round(precision_score(y, y_pred, zero_division=0), 4),
            'Recall':    round(recall_score(y, y_pred,    zero_division=0), 4),
            'F1':        round(f1_score(y, y_pred,        zero_division=0), 4)}

lr_metrics = evaluate('Logistic Regression', lr, X_test_s, y_test)
rf_metrics = evaluate('Random Forest',       rf, X_test,   y_test)
gb_metrics = evaluate('Gradient Boosting', gb, X_test_s, y_test)

cmp_df = pd.DataFrame([lr_metrics, rf_metrics, gb_metrics]).sort_values('F1', ascending=False)
print(cmp_df.to_string(index=False))


              Model  Accuracy  Precision  Recall     F1
  Gradient Boosting    0.9958     0.9945  1.0000 0.9972
      Random Forest    0.9917     0.9890  1.0000 0.9945
Logistic Regression    0.9292     0.9711  0.9333 0.9518


## 6. Pick Best Model & Score All Customers

In [19]:
best_metrics = max([lr_metrics, rf_metrics, gb_metrics], key=lambda x: x['F1'])
if best_metrics['Model'] == 'Logistic Regression':
    best_model, use_scaler, best_name = lr, True, 'Logistic Regression'
elif best_metrics['Model'] == 'Random Forest':
    best_model, use_scaler, best_name = rf, False, 'Random Forest'
else:
    best_model, use_scaler, best_name = gb, True, 'Gradient Boosting'
print(f'Best model: {best_name}')

X_all         = scaler.transform(X) if use_scaler else X
risk_scores   = best_model.predict_proba(X_all)[:, 1]

customer_df['risk_score'] = risk_scores.round(4)
customer_df['risk_class'] = (risk_scores > 0.7).astype(int)

print(f'\nRisk score range: {risk_scores.min():.4f} – {risk_scores.max():.4f}')
print(f'High-risk (>0.7): {customer_df["risk_class"].sum()}')
customer_df[['CustomerID','risk_score','risk_class']].head()


Best model: Gradient Boosting

Risk score range: 0.0000 – 1.0000
High-risk (>0.7): 899


,CustomerID,risk_score,risk_class
0,CUST100000,1.0,1
1,CUST100001,0.0,0
2,CUST100002,1.0,1
3,CUST100003,1.0,1
4,CUST100004,1.0,1


In [22]:
df.head()

,CustomerID,CustomerName,TransactionID,Date,Description,Amount,Balance,Merchant,TransactionType,is_subscription,Status,Frequency,Is_Recurring
0,CUST100000,Karan Verma,TXN28728463,2023-01-02,MONTHLY SALARY CREDIT,143839.29,223750.99,Employer,CREDIT,0,SUCCESS,monthly,0
1,CUST100000,Karan Verma,TXN31429110,2023-01-14,NETFLIX.COM MONTHLY SUB,661.05,NaN,Netflix,DEBIT,1,SUCCESS,monthly,1
2,CUST100000,Karan Verma,TXN23756669,2023-02-02,MONTHLY SALARY CREDIT,143839.29,367590.28,Employer,CREDIT,0,SUCCESS,monthly,0
3,CUST100000,Karan Verma,TXN55176955,2023-02-14,NETFLIX.COM MONTHLY SUB,654.12,2667703.75,Netflix,DEBIT,1,SUCCESS,monthly,1
4,CUST100000,Karan Verma,TXN83197857,2023-03-02,MONTHLY SALARY CREDIT,143839.29,511429.57,Employer,CREDIT,0,SUCCESS,monthly,0


In [23]:
# --- derive same features as fr6_risk_scoring.py from raw data ---
df["Is_Recurring"] = df["is_subscription"].astype(int)

# convert existing frequency strings to a column that can be used for monthly logic
df["Inferred_Freq"] = df["Frequency"].fillna("irregular").str.capitalize()

cust = df.groupby("CustomerID").agg(
    Current_Balance = ("Balance", "last"),
    Failed_Debits   = ("Status", lambda s: (s == "FAILED").sum()),
    Total_Debits    = ("TransactionType", lambda s: (s == "DEBIT").sum()),
    Avg_Debit_Amount = ("Amount", lambda a: a[df.loc[a.index, "TransactionType"] == "DEBIT"].mean()),
    Total_Monthly_Sub_Amount = ("Amount",
        lambda a: a[(df.loc[a.index, "Is_Recurring"] == 1) & (df.loc[a.index, "Inferred_Freq"] == "Monthly")].sum()),
    Subscription_Count = ("Description",
        lambda x: x[(df.loc[x.index, "Is_Recurring"] == 1) & (df.loc[x.index, "Inferred_Freq"] == "Monthly")].nunique()),
).reset_index()

cust["Failed_Debit_Rate"] = (cust["Failed_Debits"] / cust["Total_Debits"].replace(0, 1)).clip(0, 1)

# for notebook evidence, we use placeholder upcoming debit (FR5 output later overrides this)
cust["Upcoming_Total_Debit"]    = 1.0
cust["Balance_To_Debit_Ratio"] = (cust["Current_Balance"] / cust["Upcoming_Total_Debit"]).clip(0, 20)
cust["Upcoming_Debit_Pct"]     = (cust["Upcoming_Total_Debit"] / cust["Current_Balance"].replace(0, 1)).clip(0, 5)

FEATURES = [
    "Current_Balance", "Failed_Debit_Rate", "Avg_Debit_Amount",
    "Upcoming_Total_Debit", "Balance_To_Debit_Ratio",
    "Upcoming_Debit_Pct", "Total_Monthly_Sub_Amount", "Subscription_Count",
]

cust = cust.fillna(0)
print(cust[FEATURES].head())

   Current_Balance  Failed_Debit_Rate  Avg_Debit_Amount  Upcoming_Total_Debit  \
0       2652305.97           0.000000       2197.129118                   1.0   
1       1126975.69           0.028169       2105.444493                   1.0   
2       1345594.07           0.043478        737.095225                   1.0   
3       2134585.12           0.030075       1378.179008                   1.0   
4       2426920.94           0.065789       1220.923245                   1.0   

   Balance_To_Debit_Ratio  Upcoming_Debit_Pct  Total_Monthly_Sub_Amount  \
0                    20.0        3.770304e-07                  11621.20   
1                    20.0        8.873306e-07                  73160.00   
2                    20.0        7.431662e-07                   2330.82   
3                    20.0        4.684751e-07                  16325.13   
4                    20.0        4.120447e-07                 114606.14   

   Subscription_Count  
0                   1  
1             

In [24]:

# sample label logic (or existing y from your notebook risk_label)
cust["risk_label"] = ((cust["Failed_Debits"] > 2) | 
                      (cust["Current_Balance"] > cust["Current_Balance"].quantile(0.9))).astype(int)

X = cust[FEATURES]
y = cust["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler     = StandardScaler().fit(X_train)
X_train_s  = scaler.transform(X_train)
X_test_s   = scaler.transform(X_test)

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.07, subsample=0.8, random_state=42)
gb.fit(X_train_s, y_train)
print("GB accuracy:", accuracy_score(y_test, gb.predict(X_test_s)))
print("GB F1:", f1_score(y_test, gb.predict(X_test_s)))

GB accuracy: 0.9708333333333333
GB F1: 0.978978978978979


In [25]:
importances = gb.feature_importances_
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(importance_df)

                    feature  importance
0         Failed_Debit_Rate    0.634508
1          Avg_Debit_Amount    0.113462
2        Subscription_Count    0.096281
3        Upcoming_Debit_Pct    0.067374
4           Current_Balance    0.053452
5  Total_Monthly_Sub_Amount    0.034923
6      Upcoming_Total_Debit    0.000000
7    Balance_To_Debit_Ratio    0.000000
